# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have a limited chances of making queries in total.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

Set a Random Seed

In [ ]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

## 1. collect training data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [ ]:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [ ]:
len(sequence_wt)

656

In [ ]:
def get_mutated_sequence(mut, sequence_wt):
  wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

  sequence = deepcopy(sequence_wt)

  return sequence[:pos]+mt+sequence[pos+1:]

In [ ]:
df_train = pd.read_csv('train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [ ]:
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


### Quick EDA
**Exploratory Data Analysis to understand fitness distribution and mutation coverage**

In [ ]:
print("Train size:", len(df_train))
print("Test size:", len(df_test))

print("\nFitness stats:")
print(df_train['DMS_score'].describe())

positions = df_train['mutant'].apply(lambda x: int(x[1:-1]))

plt.figure(figsize=(6, 4))
plt.hist(df_train['DMS_score'], bins=50)
plt.title("Fitness Score Distribution")
plt.xlabel("DMS_score")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(positions, bins=50)
plt.title("Mutation Position Distribution")
plt.xlabel("Position")
plt.ylabel("Count")
plt.show()

print("\nMissing values in train:")
print(df_train.isnull().sum())
print("\nMissing values in test:")
print(df_test.isnull().sum())

In [ ]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a prediction model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

In [ ]:
'''hyperparameters'''

seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

In [ ]:
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

def encode_mutation(mut, seq_length=656):
    wt_aa, pos, mut_aa = mut[0], int(mut[1:-1]), mut[-1]
    vec = []
    # Position (normalized)
    vec.append(pos / seq_length)
    # WT amino acid (one-hot, 20)
    wt_oh = [0.0] * 20
    if wt_aa in aa_to_idx: wt_oh[aa_to_idx[wt_aa]] = 1.0
    vec.extend(wt_oh)
    # Mutant amino acid (one-hot, 20)
    mt_oh = [0.0] * 20
    if mut_aa in aa_to_idx: mt_oh[aa_to_idx[mut_aa]] = 1.0
    vec.extend(mt_oh)

    return np.array(vec, dtype=np.float32)

class ProteinDataset(Dataset):
    def __init__(self, df, istrain=True):
        self.encodings = np.stack([encode_mutation(m) for m in df['mutant'].values])
        self.targets = df['DMS_score'].values.astype(np.float32) if istrain else np.zeros(len(df), dtype=np.float32)

    def __len__(self): return len(self.encodings)
    def __getitem__(self, idx): return torch.tensor(self.encodings[idx]), torch.tensor(self.targets[idx])

In [ ]:
train_dataset = ProteinDataset(df_train)
test_dataset = ProteinDataset(df_test, istrain=False)

# split validation set
train_dataset, val_dataset = train_test_split(train_dataset, test_size=val_ratio, random_state=seed, shuffle=True)

# TODO: revise according to your own model
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from scipy.stats import spearmanr
import numpy as np

X_train = np.stack([x[0].numpy() for x in train_dataset])
y_train = np.array([x[1].numpy() for x in train_dataset])
X_val   = np.stack([x[0].numpy() for x in val_dataset])
y_val   = np.array([x[1].numpy() for x in val_dataset])
X_test  = np.stack([x[0].numpy() for x in test_dataset])

# Combine train and val for final training to get maximum data
X_full_train = np.concatenate([X_train, X_val])
y_full_train = np.concatenate([y_train, y_val])

# Train a Gradient Boosting Regressor and evaluate on validation set
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
gbr.fit(X_train, y_train)

val_preds = gbr.predict(X_val)
val_spearman = spearmanr(y_val, val_preds)[0]
print(f"Gradient Boosting val Spearman: {val_spearman:.4f}")

# Retrain on full training data for final test predictions
gbr.fit(X_full_train, y_full_train)
y_test_pred = gbr.predict(X_test)
print("Predictions generated for test set.")

Gradient Boosting val Spearman: 0.4065
Predictions generated for test set.


In [ ]:
df_test['DMS_score_predicted'] = y_test_pred
df_test

,mutant,sequence,DMS_score_predicted
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.300281
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.292661
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.416386
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.387519
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.281896
...,...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.333692
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.405352
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.351825
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.478202


In [ ]:
#df_test['id'] = df_test.index
#df_test[['id', 'DMS_score_predicted']].rename(columns={'DMS_score_predicted': 'DMS_score'}).to_csv('predictions.csv', index=False)
df_test[['mutant', 'DMS_score_predicted']].to_csv('predictions.csv', index=False)

## 3. Select query for next round

In [ ]:
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted
2386,Q132D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.847843
2133,Y118F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.827412
8868,Q526D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.814210
8059,Q484D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.814210
9401,Q554D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.814210
...,...,...,...
8264,D494E,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.658373
9556,I562L,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.656442
8290,I496L,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.656442
10479,I611L,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.656442


In [ ]:
best_mutants = df_test.sort_values('DMS_score_predicted', ascending=False).head(10)['mutant'].values
with open('top10.txt', 'w') as f:
    for mut in best_mutants:
        f.write(mut + '\n')
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted
2386,Q132D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.847843
2133,Y118F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.827412
8868,Q526D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.814210
8059,Q484D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.814210
9401,Q554D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.814210
...,...,...,...
8264,D494E,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.658373
9556,I562L,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.656442
8290,I496L,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.656442
10479,I611L,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.656442


In [ ]:
# Example: randomly select 100 test variants to be queried.
# Note: random selection may not be a good strategy
# TODO: select query mutants for the next round based on your own criteria

querys = np.random.choice(df_test.mutant.values, size=100, replace=False)
querys


array(['H140F', 'R387A', 'T248M', 'N643H', 'I283L', 'A144D', 'R387H',
       'D288E', 'S439R', 'H41N', 'T248W', 'S19V', 'N385H', 'K280D',
       'P458K', 'A4V', 'Q476V', 'G55R', 'G260S', 'D209I', 'A241P',
       'Q476P', 'M419S', 'L270S', 'S29E', 'W582F', 'N486A', 'T450P',
       'R265H', 'G199K', 'G120E', 'P495F', 'V540M', 'A232E', 'W94S',
       'V542A', 'G6W', 'T520Q', 'L620P', 'P491R', 'D190F', 'A291L',
       'A203M', 'I568L', 'Y118R', 'F616E', 'K198N', 'T480K', 'A649N',
       'S429I', 'E53F', 'T516Y', 'A301E', 'F59I', 'N488K', 'L270V',
       'F365A', 'D494L', 'F579D', 'S8D', 'E15K', 'N556Q', 'R518W', 'A69W',
       'E532M', 'N443F', 'W94L', 'I607Q', 'P363F', 'T321Y', 'L533M',
       'T128G', 'G204K', 'G35Y', 'M580S', 'G504E', 'L639E', 'L639I',
       'E432K', 'R605G', 'Y188P', 'D233H', 'V1G', 'I568V', 'R40Y',
       'L149V', 'P555R', 'L61I', 'A523N', 'K543H', 'W560G', 'S429C',
       'N629P', 'Q484W', 'S393L', 'D99A', 'F579E', 'S439C', 'E585T',
       'T248Y'], dtype=object)

In [ ]:
with open('query.txt', 'w') as f:
  for mutant in querys:
    f.write(mutant+'\n')